# Evaluation

In [1]:
import os

import pandas as pd
from datasets import load_dataset
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from trl import SFTConfig
import json
import evaluate
import numpy as np
from tqdm import tqdm
from peft import AutoPeftModelForCausalLM

set_seed(42)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

C:\Users\Marlow\anaconda3\envs\MA_New_Install\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


# MMLU Evaluation
Evaluation of the MMLU general knowledge dataset

In [2]:
mmlu_ds = load_dataset('cais/mmlu', 'all')
mmlu_ds

DatasetDict({
    test: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 14042
    })
    validation: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 1531
    })
    dev: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 285
    })
    auxiliary_train: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 99842
    })
})

In [3]:
mmlu_ds_dev = mmlu_ds['dev']
mmlu_ds_dev

Dataset({
    features: ['question', 'subject', 'choices', 'answer'],
    num_rows: 285
})

In [4]:
mmlu_ds_test = mmlu_ds['test']
mmlu_ds_test

Dataset({
    features: ['question', 'subject', 'choices', 'answer'],
    num_rows: 14042
})

In [5]:
# load the 5 shot prompts and place them at the beginning of the chat 
def get_promt_5_shot(few_shot_set, test_example):
    answer_letters = ['A', 'B', 'C', 'D']
    # few_shot_examples = few_shot_set.filter(lambda x: x['subject'] == category)
    
    messages = [
        {"role": "system", "content": "You are a multiple choice answer assistant. You only reply with the Letter (A, B, C or D), without any explanations."}
    ]
    
    for example in few_shot_set:
        messages.append({"role": "user", 
                         "content": f"Question: {example['question']}\n" 
                                    f"A: {example['choices'][0]} B: {example['choices'][1]} C: {example['choices'][2]} D: {example['choices'][3]}\n"})
        messages.append({"role": "assistant", "content": answer_letters[example['answer']]})

    messages.append({"role": "user", 
                     "content": f"Question: {test_example['question']}\n"
                                f"A: {test_example['choices'][0]} B: {test_example['choices'][1]} C: {test_example['choices'][2]} D: {test_example['choices'][3]}\n"})
    
    return messages
        

In [6]:
# # category = 'professional_law'
# # category = 'abstract_algebra'
# # category = 'professional_psychology'
# # category = 'moral_scenarios'
# # category = 'anatomy'
# # batch_size = 4
# 
# few_shot_examples = mmlu_ds_dev.filter(lambda x: x['subject'] == category)
# test_examples = mmlu_ds_test.filter(lambda x: x['subject'] == category)
# 
# messages = [get_promt_5_shot(few_shot_examples, test_ex) for test_ex in test_examples]
# 
# batched_messages = [messages[i:i + batch_size] for i in range(0, len(messages), batch_size)]
# 
# len(messages)
# 
# # batched_messages[0]

In [7]:
# evaluation pipeline for the mmlu dataset
# each category is evaluated on its own
# different batch sized tuned for my local hardware, different values do not change the results only the time to compute  
def evaluate_category(category, few_shot_set, test_set, model, tokenizer, batch_size=1):
    torch.cuda.empty_cache()
    few_shot_examples = few_shot_set[category]
    test_examples = test_set[category]
    
    messages = [get_promt_5_shot(few_shot_examples, test_ex) for test_ex in test_examples]
    if len(messages) < 500:
        batch_size = 16
    elif len(messages) < 650:
        batch_size = 8
    else:
        batch_size = 4
    if category == 'college_medicine':
        batch_size = 4
    if category == 'college_computer_science':
        batch_size = 4
    if category == 'professional_medicine':
        batch_size = 4
    if category == 'professional_law':
        batch_size = 1
    if category == 'professional_accounting':
        batch_size = 8
    if category == 'high_school_world_history':
        batch_size = 4
    if category == 'high_school_us_history':
        batch_size = 4
    if category == 'high_school_computer_science':
        batch_size = 4
    if category == 'high_school_european_history':
        batch_size = 1
    if category == 'high_school_statistics':
        batch_size = 8
    if category == 'security_studies':
        batch_size = 1


    # print(f"Batch size: {batch_size}, Length: {len(messages)}, {category}")
    
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, batch_size=batch_size)
    
    outputs = pipe(messages, max_new_tokens=1, return_full_text=False)
    
    correct = 0
    total = len(test_examples)
    i = 0
    
    for example in test_examples:
        answer_letters = ['A', 'B', 'C', 'D']
        correct_answer = answer_letters[example['answer']]
        if correct_answer == outputs[i][0]['generated_text'].strip():
            correct += 1
        i += 1 
        
    accuracy = correct / total
    
    return accuracy


In [7]:
def create_filtered_examples(category, dataset):
    filtered_examples = dataset.filter(lambda x: x['subject'] == category)
    return filtered_examples

In [8]:
categories = list(set(mmlu_ds_test['subject']))
categories.sort()

dev_examples = {}
test_examples = {}
for category in categories:
    dev_examples[category] = create_filtered_examples(category, mmlu_ds_dev)
    test_examples[category] = create_filtered_examples(category, mmlu_ds_test)

In [9]:
test_examples

{'abstract_algebra': Dataset({
     features: ['question', 'subject', 'choices', 'answer'],
     num_rows: 100
 }),
 'anatomy': Dataset({
     features: ['question', 'subject', 'choices', 'answer'],
     num_rows: 135
 }),
 'astronomy': Dataset({
     features: ['question', 'subject', 'choices', 'answer'],
     num_rows: 152
 }),
 'business_ethics': Dataset({
     features: ['question', 'subject', 'choices', 'answer'],
     num_rows: 100
 }),
 'clinical_knowledge': Dataset({
     features: ['question', 'subject', 'choices', 'answer'],
     num_rows: 265
 }),
 'college_biology': Dataset({
     features: ['question', 'subject', 'choices', 'answer'],
     num_rows: 144
 }),
 'college_chemistry': Dataset({
     features: ['question', 'subject', 'choices', 'answer'],
     num_rows: 100
 }),
 'college_computer_science': Dataset({
     features: ['question', 'subject', 'choices', 'answer'],
     num_rows: 100
 }),
 'college_mathematics': Dataset({
     features: ['question', 'subject', 'choic

In [10]:
model_name = 'microsoft/Phi-3-mini-128k-instruct'
baseline_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2", device_map=device)
baseline_tokenizer = AutoTokenizer.from_pretrained(model_name)


Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.22s/it]


In [11]:
results = {}
for category in tqdm(categories):
    accuracy = evaluate_category(category = category, 
                                 few_shot_set = dev_examples,
                                 test_set = test_examples,
                                 model = baseline_model,
                                 tokenizer = baseline_tokenizer,
                                 batch_size = 16)
    results[category] = accuracy
    # print(f"{category}: {accuracy}")
    
average_accuracy = sum(results.values()) / len(results)
results['total_accuracy_mean'] = average_accuracy
file = open('results/MMLU/baseline.json', 'w')
json.dump(results, file)
file.close()
print(f"Average accuracy: {average_accuracy}")

100%|██████████| 57/57 [29:39<00:00, 31.22s/it] 


FileNotFoundError: [Errno 2] No such file or directory: 'results/MMLU/baseline.json'

In [13]:
results

{'abstract_algebra': 0.37,
 'anatomy': 0.6296296296296297,
 'astronomy': 0.7302631578947368,
 'business_ethics': 0.69,
 'clinical_knowledge': 0.7660377358490567,
 'college_biology': 0.8333333333333334,
 'college_chemistry': 0.54,
 'college_computer_science': 0.54,
 'college_mathematics': 0.37,
 'college_medicine': 0.6705202312138728,
 'college_physics': 0.43137254901960786,
 'computer_security': 0.74,
 'conceptual_physics': 0.6212765957446809,
 'econometrics': 0.49122807017543857,
 'electrical_engineering': 0.6,
 'elementary_mathematics': 0.5185185185185185,
 'formal_logic': 0.4365079365079365,
 'global_facts': 0.42,
 'high_school_biology': 0.8129032258064516,
 'high_school_chemistry': 0.5960591133004927,
 'high_school_computer_science': 0.66,
 'high_school_european_history': 0.7515151515151515,
 'high_school_geography': 0.7828282828282829,
 'high_school_government_and_politics': 0.8652849740932642,
 'high_school_macroeconomics': 0.7435897435897436,
 'high_school_mathematics': 0.322222

In [10]:
finetuned_model_name = "./finetuned_phi-3/"
finetuned_model = AutoModelForCausalLM.from_pretrained(finetuned_model_name,
                                                       torch_dtype=torch.bfloat16,
                                                       trust_remote_code=True,
                                                       device_map=device,
                                                       attn_implementation="flash_attention_2",
                                                       )
finetuned_tokenizer = AutoTokenizer.from_pretrained(finetuned_model_name)
# finetuned_tokenizer.padding_side = 'left'

Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.28s/it]


In [11]:
results_finetuned = {}
for category in tqdm(categories):
    accuracy = evaluate_category(category = category,
                                 few_shot_set = dev_examples,
                                 test_set = test_examples,
                                 model = finetuned_model,
                                 tokenizer = finetuned_tokenizer,
                                 batch_size = 16)
    results_finetuned[category] = accuracy
    # print(f"{category}: {accuracy}")

average_accuracy = sum(results_finetuned.values()) / len(results_finetuned)
results_finetuned['total_accuracy_mean'] = average_accuracy
file = open('results/MMLU/finetuned.json', 'w')
json.dump(results_finetuned, file)
file.close()
print(f"Average accuracy (finetuned): {average_accuracy}")

100%|██████████| 57/57 [34:47<00:00, 36.63s/it] 

Average accuracy (finetuned): 0.6454397574671737


In [13]:
results_finetuned

{'abstract_algebra': 0.34,
 'anatomy': 0.6222222222222222,
 'astronomy': 0.7236842105263158,
 'business_ethics': 0.66,
 'clinical_knowledge': 0.7471698113207547,
 'college_biology': 0.7986111111111112,
 'college_chemistry': 0.52,
 'college_computer_science': 0.53,
 'college_mathematics': 0.39,
 'college_medicine': 0.6878612716763006,
 'college_physics': 0.45098039215686275,
 'computer_security': 0.73,
 'conceptual_physics': 0.5872340425531914,
 'econometrics': 0.47368421052631576,
 'electrical_engineering': 0.5655172413793104,
 'elementary_mathematics': 0.4973544973544973,
 'formal_logic': 0.40476190476190477,
 'global_facts': 0.36,
 'high_school_biology': 0.8032258064516129,
 'high_school_chemistry': 0.5960591133004927,
 'high_school_computer_science': 0.68,
 'high_school_european_history': 0.6909090909090909,
 'high_school_geography': 0.7828282828282829,
 'high_school_government_and_politics': 0.8601036269430051,
 'high_school_macroeconomics': 0.7410256410256411,
 'high_school_mathem

In [20]:
with open("./results/MMLU/baseline.json") as f:
    baseline_scores = json.load(f)
# baseline_scores

In [21]:
df1 = pd.DataFrame(baseline_scores, index=['baseline']).transpose()
# df1

In [22]:
with open("./results/MMLU/finetuned.json") as f:
    finetuned_scores = json.load(f)
# finetuned_scores

In [27]:
df2 = pd.DataFrame(finetuned_scores, index=['finetuned']).transpose()
res = pd.concat([df1, df2], axis=1)
res.drop(['total_accuracy_mean'], inplace=True)
res.round(3)

,baseline,finetuned
abstract_algebra,0.370,0.340
anatomy,0.630,0.622
astronomy,0.730,0.724
business_ethics,0.690,0.660
clinical_knowledge,0.766,0.747
college_biology,0.833,0.799
college_chemistry,0.540,0.520
college_computer_science,0.540,0.530
college_mathematics,0.370,0.390
college_medicine,0.671,0.688


In [29]:
res.describe().round(3)

,baseline,finetuned
count,57.000,57.000
mean,0.660,0.645
std,0.143,0.141
min,0.322,0.340
25%,0.540,0.528
50%,0.717,0.691
75%,0.760,0.747
max,0.885,0.885


# OceanBench Evaluation
Evaluation of the specialized knowledge OceanBench dataset


In [2]:
model_name = 'microsoft/Phi-3-mini-128k-instruct'
baseline_model = AutoModelForCausalLM.from_pretrained(model_name, 
                                                      torch_dtype=torch.bfloat16,
                                                      attn_implementation="flash_attention_2",
                                                      device_map=device)
baseline_tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.18s/it]


In [3]:
# loading lora checkpoint and merging it into the model to create a savefile for the finetuned model

# finetuned_model_name = 'G:\\Masterarbeit\\phi-3_finetuned_lora_epoch_2'
# new_model = AutoPeftModelForCausalLM.from_pretrained(
#     finetuned_model_name,
#     # low_cpu_mem_usage=True,
#     # return_dict=True,
#     torch_dtype=torch.bfloat16,
#     trust_remote_code=True,
#     device_map=device,
#     attn_implementation="flash_attention_2",
# )
# new_path = "./finetuned_phi-3/"
# finetuned_model = new_model.merge_and_unload()
# finetuned_model.save_pretrained(new_path)
# finetuned_tokenizer = AutoTokenizer.from_pretrained(finetuned_model_name)
# finetuned_tokenizer.padding_side = 'left'
# finetuned_tokenizer.save_pretrained(new_path)

Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.50s/it]


('./finetuned_phi-3/tokenizer_config.json',
 './finetuned_phi-3/special_tokens_map.json',
 './finetuned_phi-3/tokenizer.json')

In [3]:
finetuned_model_name = "./finetuned_phi-3/"
finetuned_model = AutoModelForCausalLM.from_pretrained(finetuned_model_name,
                                                       torch_dtype=torch.bfloat16,
                                                       trust_remote_code=True,
                                                       device_map=device,
                                                       attn_implementation="flash_attention_2",
                                                       )
finetuned_tokenizer = AutoTokenizer.from_pretrained(finetuned_model_name)
# finetuned_tokenizer.padding_side = 'left'

Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.71s/it]


In [2]:
#loading dataset and metrics
ocean_bench_ds = load_dataset('zjunlp/OceanBench', split='train')
rouge_metric = evaluate.load("rouge", seed=42)
bleu_metric = evaluate.load("bleu", seed=42)
meteor_metric = evaluate.load("meteor", seed=42)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Marlow\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Marlow\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Marlow\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
ocean_bench_ds

Dataset({
    features: ['text'],
    num_rows: 10344
})

In [4]:
ocean_bench_ds[0]['text']

'{"task_type": "Analysis", "input": "Analyze the impact of marine environmental dynamics processes on the wave environment.", "output": "The dynamic processes of the oceanic environment have a significant influence on the wave conditions, such as the temperature of seawater, salinity, and ocean current affecting the height and frequency of waves. Meanwhile, the wave conditions can further impact the dynamics of ocean current and tides."}'

In [5]:
def format_data(example):
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."}
    ]
    
    entry = json.loads(example['text'])
    messages.append({"role": "user", "content": entry['input']})
    
    # return {'messages': messages}   
    return messages

In [6]:
# evaluation of the OceanBench dataset
def evaluate_dataset(test_set, model, tokenizer, batch_size = 1, max_new_tokens = 512):
    torch.cuda.empty_cache()
    
    messages = [format_data(entry) for entry in tqdm(test_set, desc="Preparing data")]
    start_time = datetime.datetime.now()
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, batch_size=batch_size)
    
    #creating models responses 
    with torch.no_grad():
        outputs = []
        for i in tqdm(range(0, len(messages), batch_size), desc="Generating responses"):
            torch.cuda.empty_cache()
            batch = messages[i:i+batch_size]
            output_batch = pipe(batch, max_new_tokens=max_new_tokens, return_full_text=False)
            outputs.extend(output_batch)
        # outputs = pipe(messages, max_new_tokens=max_new_tokens, return_full_text=False)
    
    print(f"Dauer Pipeline: {datetime.datetime.now() - start_time}")
    
    
    #extracting the ground truth answers out of the dataset
    i = 0
    preds = []
    ground_truths = []
    start_time_json = datetime.datetime.now()
    for example in tqdm(test_set, desc="Processing outputs"):
        entry = json.loads(example['text'])
        ground_truths.append(entry['output'])
        preds.append(outputs[i][0]['generated_text'])
        i += 1
    
    print(f"Dauer JSON-Schleife: {datetime.datetime.now() - start_time_json}")
    
    start_time_scores = datetime.datetime.now()
    
    bleu_score_per_entry = []
    bleu_score_n1_per_entry = []
    bleu_score_n2_per_entry = []
    meteor_score_per_entry = []
    
    #computing the scores by comparing the ground truth with the respons of the model (according to each metrics formula)
    #first loop to compute the score for each dataset entry on its own, afterwards it will be calculated over the full dataset
    for x in tqdm(range(len(preds)), desc="Computing scores per entry"):
        bleu_score_per_entry.append(bleu_metric.compute(predictions=[preds[x]], references=[ground_truths[x]]))
        bleu_score_n1_per_entry.append(bleu_metric.compute(predictions=[preds[x]], references=[ground_truths[x]], max_order=1))
        bleu_score_n2_per_entry.append(bleu_metric.compute(predictions=[preds[x]], references=[ground_truths[x]], max_order=2))
        meteor_score_per_entry.append(meteor_metric.compute(predictions=[preds[x]], references=[ground_truths[x]]))
        
    
    rouge_score_aggregated = rouge_metric.compute(predictions=preds, references=ground_truths, use_aggregator=True)
    rouge_score = rouge_metric.compute(predictions=preds, references=ground_truths, use_aggregator=False)
    bleu_score = bleu_metric.compute(predictions=preds, references=ground_truths)
    bleu_score_n1 = bleu_metric.compute(predictions=preds, references=ground_truths, max_order=1)
    bleu_score_n2 = bleu_metric.compute(predictions=preds, references=ground_truths, max_order=2)
    meteor_score = meteor_metric.compute(predictions=preds, references=ground_truths)
    
    results = {
        "rouge_aggregated": rouge_score_aggregated,
        "rouge_not_aggregated": rouge_score,
        "bleu": bleu_score,
        "bleu_n1": bleu_score_n1,
        "bleu_n2": bleu_score_n2,
        "meteor": meteor_score,
        "bleu_per_entry": bleu_score_per_entry,
        "bleu_n1_per_entry": bleu_score_n1_per_entry,
        "bleu_n2_per_entry": bleu_score_n2_per_entry,
        "meteor_per_entry": meteor_score_per_entry,
        
    }
    print(f"Dauer Scores: {datetime.datetime.now() - start_time_scores}")    
    return results

In [7]:
model_name = 'microsoft/Phi-3-mini-128k-instruct'
baseline_model = AutoModelForCausalLM.from_pretrained(model_name,
                                                      torch_dtype=torch.bfloat16,
                                                      attn_implementation="flash_attention_2",
                                                      device_map=device)
baseline_tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.23s/it]


In [8]:
baseline_out = evaluate_dataset(ocean_bench_ds, baseline_model, baseline_tokenizer, batch_size = 16, max_new_tokens = 256)
file = open(f'results/OceanBench_2/Baseline_256Token_16BatchSize.json', 'w')
json.dump(baseline_out, file)
file.close()

Generating responses:   0%|          | 3/647 [01:14<4:25:31, 24.74s/it]


KeyboardInterrupt: 

In [9]:
# baseline_model.to('cpu')
del baseline_model
torch.cuda.empty_cache()

In [10]:
finetuned_model_name = "./finetuned_phi-3/"
finetuned_model = AutoModelForCausalLM.from_pretrained(finetuned_model_name,
                                                       torch_dtype=torch.bfloat16,
                                                       trust_remote_code=True,
                                                       device_map=device,
                                                       attn_implementation="flash_attention_2",
                                                       )
finetuned_tokenizer = AutoTokenizer.from_pretrained(finetuned_model_name)

Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.58s/it]


In [11]:
finetuned_out = evaluate_dataset(ocean_bench_ds, finetuned_model, finetuned_tokenizer, batch_size = 16, max_new_tokens = 256)
file = open(f'results/OceanBench_2/Finetuned_256Token_16BatchSize.json', 'w')
json.dump(finetuned_out, file)
file.close()

Generating responses: 100%|██████████| 324/324 [1:00:37<00:00, 11.23s/it]


Dauer Pipeline: 1:00:37.140401


Processing outputs: 100%|██████████| 10344/10344 [00:00<00:00, 38954.53it/s]


Dauer JSON-Schleife: 0:00:00.265541


Computing scores per entry: 100%|██████████| 10344/10344 [05:35<00:00, 30.86it/s]


Dauer Scores: 0:07:08.047084


In [12]:
# finetuned_model.to('cpu')
del finetuned_model
torch.cuda.empty_cache()

In [9]:
# 10 Times evaluation for repeatability
results = []
for i in range(10):
    print(f"Calculating Metrics {i+1} of {10}:")
    out = evaluate_dataset(ocean_bench_ds, baseline_model, baseline_tokenizer, batch_size = 32, max_new_tokens = 128)
    results.append(out)

file = open('results/repeatability_test.json', 'w')
json.dump(results, file)
    

Calculating Metrics 1 of 10:


Generating responses: 100%|██████████| 324/324 [58:54<00:00, 10.91s/it]


Dauer Pipeline: 0:58:54.777516


Processing outputs: 100%|██████████| 10344/10344 [00:00<00:00, 38950.33it/s]


Dauer JSON-Schleife: 0:00:00.265569
Dauer Scores: 0:01:34.705077
Calculating Metrics 2 of 10:


...

Dauer JSON-Schleife: 0:00:00.265588
Dauer Scores: 0:01:27.564998
Calculating Metrics 10 of 10:


Generating responses: 100%|██████████| 324/324 [57:46<00:00, 10.70s/it]


Dauer Pipeline: 0:57:46.309020


Processing outputs: 100%|██████████| 10344/10344 [00:00<00:00, 38905.97it/s]


Dauer JSON-Schleife: 0:00:00.265871
Dauer Scores: 0:01:27.247139


In [29]:
for r in results:
    rouge1 = r['rouge_aggregated']['rouge1']
    rouge2 = r['rouge_aggregated']['rouge2']
    rougeL = r['rouge_aggregated']['rougeL']
    rougeLsum = r['rouge_aggregated']['rougeLsum']

    rouge1_not_aggregated = r['rouge_not_aggregated']['rouge1']
    rouge2_not_aggregated = r['rouge_not_aggregated']['rouge2']
    rougeL_not_aggregated = r['rouge_not_aggregated']['rougeL']
    rougeLsum_not_aggregated = r['rouge_not_aggregated']['rougeLsum']
    
    bleu_score = r['bleu']['bleu']
    bleu_precisions = r['bleu']['precisions']
    bleu_brevity_penalty = r['bleu']['brevity_penalty']
    bleu_length_ratio = r['bleu']['length_ratio']
    
    meteor_score = r['meteor']['meteor']
    
    # print(f"{rouge1} <-- aggregated\n{np.mean(rouge1_not_aggregated)} <-- mean of not aggregated\n{np.median(rouge1_not_aggregated)} <-- median of not aggregated\n")
    # print(f"{bleu_score}\n{bleu_precisions}\n")


# Math Dataset Evaluation
Evaluation of the math dataset to test the models capability towards working with numbers

In [2]:
math_ds = load_dataset('BackpropBuff/simple_math_2_numbers_1m', split='train')
math_ds

Dataset({
    features: ['text', 'output'],
    num_rows: 1000000
})

In [3]:
model_name = 'microsoft/Phi-3-mini-128k-instruct'
baseline_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2", device_map=device)
baseline_tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.84s/it]


In [ ]:
finetuned_model_name = 'G:\\Masterarbeit\\phi-3_finetuned_lora_epoch_2'
new_model = AutoPeftModelForCausalLM.from_pretrained(
    finetuned_model_name,
    # low_cpu_mem_usage=True,
    # return_dict=True,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map=device,
    attn_implementation="flash_attention_2",
)
finetuned_model = new_model.merge_and_unload()
finetuned_tokenizer = AutoTokenizer.from_pretrained(finetuned_model_name)
finetuned_tokenizer.padding_side = 'left'

In [4]:
#splitting the dataset into subsets according to the arithmetic operation
# add = []
# sub = []
# div = []
# mul = []
# 
# for entry in tqdm(math_ds): 
#     if "/" in entry['text']:
#         div.append(entry)
#     elif "*" in entry['text']:
#         mul.append(entry)
#     elif "+" in entry['text']:
#         add.append(entry)
#     elif "-" in entry['text']:
#         sub.append(entry)
#         
# math_ds_addition = Dataset.from_list(add)
# math_ds_subtraction = Dataset.from_list(sub)
# math_ds_division = Dataset.from_list(div)
# math_ds_multiplication = Dataset.from_list(mul)
# 
# math_ds_addition.info.description = (f"Original Dataset: 'BackpropBuff/simple_math_2_numbers_1m'\n"
#                                      f"Addition Dataset - Subset with only additions\n")
# math_ds_subtraction.info.description = (f"Original Dataset: 'BackpropBuff/simple_math_2_numbers_1m'\n"
#                                         f"Subtraction Dataset - Subset with only subtractions\n")
# math_ds_division.info.description = (f"Original Dataset: 'BackpropBuff/simple_math_2_numbers_1m'\n"
#                                      f"division Dataset - Subset with only divisions\n")
# math_ds_multiplication.info.description = (f"Original Dataset: 'BackpropBuff/simple_math_2_numbers_1m'\n"
#                                            f"Multiplication Dataset - Subset with only multiplications\n")
# 
# math_ds_addition.save_to_disk("./datasets/math_addition")
# math_ds_subtraction.save_to_disk("./datasets/math_subtraction")
# math_ds_division.save_to_disk("./datasets/math_division")
# math_ds_multiplication.save_to_disk("./datasets/math_multiplication")
# 
# # print(math_ds_addition)
# # print(math_ds_subtraction)
# # print(math_ds_division)
# # print(math_ds_multiplication)

Saving the dataset (1/1 shards): 100%|██████████| 248858/248858 [00:00<00:00, 3185197.71 examples/s]


In [4]:
math_ds_add = Dataset.load_from_disk("./datasets/math_addition")
math_ds_sub = Dataset.load_from_disk("./datasets/math_subtraction")
math_ds_mul = Dataset.load_from_disk("./datasets/math_multiplication")
math_ds_div = Dataset.load_from_disk("./datasets/math_division")

In [5]:
def evaluate_math_dataset(data, model, tokenizer, batch_size = 1, max_new_tokens = 128):
    torch.cuda.empty_cache()
    
    messages = []
    
    for entry in tqdm(data, desc="Preparing data"):
        text = []
        text.append({"role": "system", "content": "You are a calculation assistant. You only reply with the result to the equation."})
        text.append({"role": "user", "content": f"{entry['text']}"})
        messages.append(text)

    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, batch_size=batch_size)
    # outputs = pipe(messages, max_new_tokens=max_new_tokens, return_full_text=False)

    outputs = []
    for i in tqdm(range(0, len(messages), batch_size), desc="Generating responses"):
        torch.cuda.empty_cache()
        batch = messages[i:i+batch_size]
        output_batch = pipe(batch, max_new_tokens=max_new_tokens, return_full_text=False)
        outputs.extend(output_batch)

    correct = 0
    total = len(data)
    i = 0

    for entry in tqdm(data, desc="Processing outputs"):
        correct_solution = entry['output']
        if correct_solution == outputs[i][0]['generated_text'].strip():
            correct += 1
        i += 1

    accuracy = correct / total

    return accuracy

In [6]:
# splitting the dataset by the number of negative numbers. One, two or zero negative numbers 
def split_math_ds(ds, name_of_method):
    print(f"Splitting Dataset: {name_of_method} in zero, one and two negativs")
    no_neg  = []
    one_neg = []
    two_neg = []
    if name_of_method == "subtraction":
        for entry in tqdm(ds):
            if entry['text'].count("-") > 2:
                two_neg.append(entry)
            elif entry['text'].count("-") < 2:
                no_neg.append(entry)
            else:
                one_neg.append(entry)
    else:
        for entry in tqdm(ds):
            if entry['text'].count("-") > 1:
                two_neg.append(entry)
            elif entry['text'].count("-") < 1:
                no_neg.append(entry)
            else:
                one_neg.append(entry)
        
    datasets = {"no_neg": Dataset.from_list(no_neg), 
                "one_neg": Dataset.from_list(one_neg), 
                "two_neg": Dataset.from_list(two_neg)}
    datasets["no_neg"].info.description  = f"{name_of_method} - has no negativs"
    datasets["one_neg"].info.description = f"{name_of_method} - has one negativ"
    datasets["two_neg"].info.description = f"{name_of_method} - has two negativs"
    # for key, value in datasets.items():
    #     value.save_to_disk(f"./datasets/special_cases/{name_of_method}_{key}")
    return datasets

In [7]:
ds = {"addition": math_ds_add, "subtraction": math_ds_sub, "multiplication": math_ds_mul, "division": math_ds_div}
all_sub_sets = []
for key, value in ds.items():
    all_sub_sets.append(split_math_ds(value, key))

Splitting Dataset: addition in zero, one and two negativs


100%|██████████| 250242/250242 [00:07<00:00, 35281.89it/s]


Splitting Dataset: subtraction in zero, one and two negativs


100%|██████████| 250531/250531 [00:07<00:00, 35686.41it/s]


Splitting Dataset: multiplication in zero, one and two negativs


100%|██████████| 248858/248858 [00:07<00:00, 34281.99it/s]


Splitting Dataset: division in zero, one and two negativs


100%|██████████| 250369/250369 [00:07<00:00, 35134.75it/s]


In [8]:
len(max(math_ds['output'], key=len))

23

In [9]:
for ds in all_sub_sets:
    for key, value in ds.items():
        print(f"\033[1m{value.description}\n \033[0m"
              f"{len(value)}")
        print(len(max(value['output'], key=len)))
        print(f"{value.description.split(sep=' ')[0]}_{key}")
    print("")

addition - has no negativs
 62922
7
addition_no_neg
addition - has one negativ
 124762
7
addition_one_neg
addition - has two negativs
 62558
8
addition_two_neg

subtraction - has no negativs
 62631
7
subtraction_no_neg
subtraction - has one negativ
 125380
8
subtraction_one_neg
subtraction - has two negativs
 62520
7
subtraction_two_neg

multiplication - has no negativs
 61814
12
multiplication_no_neg
multiplication - has one negativ
 124520
13
multiplication_one_neg
multiplication - has two negativs
 62524
12
multiplication_two_neg

division - has no negativs
 62344
22
division_no_neg
division - has one negativ
 125277
23
division_one_neg
division - has two negativs
 62748
22
division_two_neg



In [1]:
# evaluation of the math subsets  

accuracy_scores_all = {}
for ds in all_sub_sets:
    accuracy_scores = {}
    for key, value in ds.items():
        print(f"Evaluating:\n"
              f"{value.description}\n"
              f"Number of entries: {len(value)}")
        acc = evaluate_math_dataset(value, baseline_model, baseline_tokenizer, batch_size = 256, max_new_tokens = 23)
        print(f"Accuracy: {acc}\n")
        accuracy_scores[f"{value.description.split(sep=' ')[0]}_{key}"] = acc
    file = open(f'results/special_cases/math_ds_{value.description.split(sep=' ')[0]}_splitted.json', 'w')
    json.dump(accuracy_scores, file)
    file.close()
    print(accuracy_scores)
    accuracy_scores_all[value.description.split(sep=' ')[0]] = accuracy_scores

NameError: name 'all_sub_sets' is not defined

In [22]:
accuracy_scores_all

{'subtraction': {'subtraction_no_neg': 0.5404991138573549,
  'subtraction_one_neg': 0.3975992981336736,
  'subtraction_two_neg': 0.3081573896353167},
 'multiplication': {'multiplication_no_neg': 9.706538971753971e-05,
  'multiplication_one_neg': 0.00012849341471249598,
  'multiplication_two_neg': 7.99692917919519e-05},
 'division': {'division_no_neg': 4.8120107789041446e-05,
  'division_one_neg': 4.789386719030628e-05,
  'division_two_neg': 7.968381462357366e-05}}

In [23]:
accuracy_scores_all['addition'] = {"addition_no_neg": 0.6688439655446425, "addition_one_neg": 0.4928984787034514, "addition_two_neg": 0.6025608235557403}
accuracy_scores_all

{'subtraction': {'subtraction_no_neg': 0.5404991138573549,
  'subtraction_one_neg': 0.3975992981336736,
  'subtraction_two_neg': 0.3081573896353167},
 'multiplication': {'multiplication_no_neg': 9.706538971753971e-05,
  'multiplication_one_neg': 0.00012849341471249598,
  'multiplication_two_neg': 7.99692917919519e-05},
 'division': {'division_no_neg': 4.8120107789041446e-05,
  'division_one_neg': 4.789386719030628e-05,
  'division_two_neg': 7.968381462357366e-05},
 'addition': {'addition_no_neg': 0.6688439655446425,
  'addition_one_neg': 0.4928984787034514,
  'addition_two_neg': 0.6025608235557403}}

In [24]:
file = open('results/special_cases/math_ds_splitted.json', 'w')
json.dump(accuracy_scores_all, file)
file.close()

In [22]:
# evaluation of the math dataset with lower accuracy (fewer decimal digits) 

def evaluate_math_dataset_lower_accurary(data, model, tokenizer, batch_size = 1, max_new_tokens = 23, num_prediction_tokens = 8):
    torch.cuda.empty_cache()

    messages = []

    for entry in tqdm(data, desc="Preparing data"):
        text = []
        text.append({"role": "system", "content": "You are a calculation assistant. You only reply with the result to the equation."})
        text.append({"role": "user", "content": f"{entry['text']}"})
        messages.append(text)

    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, batch_size=batch_size)
    # outputs = pipe(messages, max_new_tokens=max_new_tokens, return_full_text=False)

    outputs = []
    for i in tqdm(range(0, len(messages), batch_size), desc="Generating responses"):
        torch.cuda.empty_cache()
        batch = messages[i:i+batch_size]
        output_batch = pipe(batch, max_new_tokens=max_new_tokens, return_full_text=False)
        outputs.extend(output_batch)

    correct = 0
    total = len(data)
    i = 0

    for entry in tqdm(data, desc="Processing outputs"):
        correct_solution = entry['output'][:num_prediction_tokens]
        if correct_solution in outputs[i][0]['generated_text'].strip():
            correct += 1
        i += 1

    accuracy = correct / total

    return accuracy

In [30]:
accuracy_scores = {}
for key, value in all_sub_sets[3].items():
    print(f"Evaluating:\n"
          f"{value.description}\n"
          f"Number of entries: {len(value)}")
    acc = evaluate_math_dataset_lower_accurary(value, baseline_model, baseline_tokenizer, batch_size = 256, num_prediction_tokens=8)
    print(f"Accuracy: {acc}\n")
    accuracy_scores[f"{value.description.split(sep=' ')[0]}_{key}"] = acc
file = open(f'results/special_cases/math_ds_{value.description.split(sep=' ')[0]}_splitted_8Tokens.json', 'w')
json.dump(accuracy_scores, file)
file.close()
print(accuracy_scores)

Evaluating:
division - has no negativs
Number of entries: 62344


Processing outputs: 100%|██████████| 62344/62344 [00:01<00:00, 37282.71it/s]


Accuracy: 0.0005934813293981779

Evaluating:
division - has one negativ
Number of entries: 125277


Processing outputs: 100%|██████████| 125277/125277 [00:03<00:00, 36427.25it/s]


Accuracy: 0.003216871412948905

Evaluating:
division - has two negativs
Number of entries: 62748


Processing outputs: 100%|██████████| 62748/62748 [00:01<00:00, 36561.03it/s]

Accuracy: 0.00047810288774144193

{'division_no_neg': 0.0005934813293981779, 'division_one_neg': 0.003216871412948905, 'division_two_neg': 0.00047810288774144193}


In [31]:
accuracy_scores = {}
for key, value in all_sub_sets[3].items():
    print(f"Evaluating:\n"
          f"{value.description}\n"
          f"Number of entries: {len(value)}")
    acc = evaluate_math_dataset_lower_accurary(value, baseline_model, baseline_tokenizer, batch_size = 256, num_prediction_tokens=5)
    print(f"Accuracy: {acc}\n")
    accuracy_scores[f"{value.description.split(sep=' ')[0]}_{key}"] = acc
file = open(f'results/special_cases/math_ds_{value.description.split(sep=' ')[0]}_splitted_5Tokens.json', 'w')
json.dump(accuracy_scores, file)
file.close()
print(accuracy_scores)

Evaluating:
division - has no negativs
Number of entries: 62344


Processing outputs: 100%|██████████| 62344/62344 [00:01<00:00, 37287.87it/s]


Accuracy: 0.22918003336327472

Evaluating:
division - has one negativ
Number of entries: 125277


Processing outputs: 100%|██████████| 125277/125277 [00:03<00:00, 36748.74it/s]


Accuracy: 0.5939877232053768

Evaluating:
division - has two negativs
Number of entries: 62748


Processing outputs: 100%|██████████| 62748/62748 [00:01<00:00, 36505.75it/s]

Accuracy: 0.21967234015426787

{'division_no_neg': 0.22918003336327472, 'division_one_neg': 0.5939877232053768, 'division_two_neg': 0.21967234015426787}


In [31]:
accuracy = evaluate_math_dataset_lower_accurary(math_ds_div, baseline_model, baseline_tokenizer, batch_size = 256, max_new_tokens = 8)

Processing outputs: 100%|██████████| 250369/250369 [00:06<00:00, 36897.97it/s]


In [32]:
accuracy

0.0016295947181959429

In [34]:
file = open('results/special_cases/math_ds_div_8Tokens.json', 'w')
json.dump(accuracy, file)
file.close()

In [6]:
# ca 5 Stunden
datasets = [math_ds_add, math_ds_sub, math_ds_mul, math_ds_div]
accuracy_scores = []
for ds in datasets:
    print(f"Evaluating:\n"
          f"{ds.description}"
          f"Number of entries: {len(ds)}")
    acc = evaluate_math_dataset(ds, baseline_model, baseline_tokenizer, batch_size = 256, max_new_tokens = 23)
    print(f"Accuracy: {acc}\n")
    accuracy_scores.append(acc)
    
accuracy_scores

Evaluating:
Original Dataset: 'BackpropBuff/simple_math_2_numbers_1m'
Addition Dataset - Subset with only additions
Number of entries: 250242



Processing outputs: 100%|██████████| 250242/250242 [00:06<00:00, 37314.05it/s]


Accuracy: 0.5645575083319346

Evaluating:
Original Dataset: 'BackpropBuff/simple_math_2_numbers_1m'
Addition Dataset - Subset with only subtractions
Number of entries: 250531



Processing outputs: 100%|██████████| 250531/250531 [00:06<00:00, 37342.49it/s]


Accuracy: 0.410995046521189

Evaluating:
Original Dataset: 'BackpropBuff/simple_math_2_numbers_1m'
Addition Dataset - Subset with only multiplications
Number of entries: 248858



Processing outputs: 100%|██████████| 248858/248858 [00:06<00:00, 36293.73it/s]


Accuracy: 0.00010849560793705647

Evaluating:
Original Dataset: 'BackpropBuff/simple_math_2_numbers_1m'
Addition Dataset - Subset with only divisions
Number of entries: 250369



Processing outputs: 100%|██████████| 250369/250369 [00:06<00:00, 36711.94it/s]

Accuracy: 5.591746582044902e-05



[0.5645575083319346,
 0.410995046521189,
 0.00010849560793705647,
 5.591746582044902e-05]

In [8]:
results = {"addition": accuracy_scores[0], 
           "subtraction": accuracy_scores[1], 
           "multiplication": accuracy_scores[2], 
           "division": accuracy_scores[3]}
results

{'addition': 0.5645575083319346,
 'subtraction': 0.410995046521189,
 'multiplication': 0.00010849560793705647,
 'division': 5.591746582044902e-05}

In [4]:
file = open('results/math_ds_accuracies.json', 'w')
json.dump(results, file)
file.close()

In [8]:
math_ds_div

NameError: name 'math_ds_div' is not defined

In [2]:
model_name = 'microsoft/Phi-3-mini-128k-instruct'
baseline_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16,
                                                      attn_implementation="flash_attention_2", device_map=device)
baseline_tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.13s/it]


In [3]:
del baseline_model
del baseline_tokenizer

In [4]:
torch.cuda.empty_cache()

In [5]:
finetuned_model_name = 'G:\\Masterarbeit\\phi-3_finetuned_lora_epoch_2'
new_model = AutoPeftModelForCausalLM.from_pretrained(
    finetuned_model_name,
    # low_cpu_mem_usage=True,
    # return_dict=True,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map=device,
    attn_implementation="flash_attention_2",
)
finetuned_model = new_model.merge_and_unload()
finetuned_tokenizer = AutoTokenizer.from_pretrained(finetuned_model_name)
finetuned_tokenizer.padding_side = 'left'

Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.53s/it]


In [9]:
del finetuned_model
del finetuned_tokenizer
del new_model

NameError: name 'finetuned_model' is not defined

In [10]:
torch.cuda.empty_cache()

# Custom Math Dataset

In [1]:
import datetime
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed, pipeline
import json
from tqdm import tqdm

set_seed(42)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

C:\Users\Marlow\anaconda3\envs\MA_New_Install\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [2]:
custom_math_ds = Dataset.load_from_disk("./datasets/custom_math_ds/train")
custom_math_ds

Dataset({
    features: ['text', 'output'],
    num_rows: 200000
})

In [3]:
# add = []
# sub = []
# div = []
# mul = []
# 
# for entry in tqdm(custom_math_ds): 
#     if "/" in entry['text']:
#         div.append(entry)
#     elif "*" in entry['text']:
#         mul.append(entry)
#     elif "+" in entry['text']:
#         add.append(entry)
#     elif "-" in entry['text']:
#         sub.append(entry)
# 
# math_ds_addition = Dataset.from_list(add)
# math_ds_subtraction = Dataset.from_list(sub)
# math_ds_division = Dataset.from_list(div)
# math_ds_multiplication = Dataset.from_list(mul)
# 
# math_ds_addition.info.description = (f"Original Dataset: 'Custom Math Dataset - Lower Bound: -1000 Upper Bound: 1000'\n"
#                                      f"Addition Dataset - Subset with only additions\n")
# math_ds_subtraction.info.description = (f"Original Dataset: 'Custom Math Dataset - Lower Bound: -1000 Upper Bound: 1000'\n"
#                                         f"Subtraction Dataset - Subset with only subtractions\n")
# math_ds_division.info.description = (f"Original Dataset: 'Custom Math Dataset - Lower Bound: -1000 Upper Bound: 1000'\n"
#                                      f"division Dataset - Subset with only divisions\n")
# math_ds_multiplication.info.description = (f"Original Dataset: 'Custom Math Dataset - Lower Bound: -1000 Upper Bound: 1000'\n"
#                                            f"Multiplication Dataset - Subset with only multiplications\n")
# 
# math_ds_addition.save_to_disk("./datasets/custom/math_addition")
# math_ds_subtraction.save_to_disk("./datasets/custom/math_subtraction")
# math_ds_division.save_to_disk("./datasets/custom/math_division")
# math_ds_multiplication.save_to_disk("./datasets/custom/math_multiplication")
# 
# # print(math_ds_addition)
# # print(math_ds_subtraction)
# # print(math_ds_division)
# # print(math_ds_multiplication)

In [4]:
math_ds_add = Dataset.load_from_disk("./datasets/custom/math_addition")
math_ds_sub = Dataset.load_from_disk("./datasets/custom/math_subtraction")
math_ds_mul = Dataset.load_from_disk("./datasets/custom/math_multiplication")
math_ds_div = Dataset.load_from_disk("./datasets/custom/math_division")

In [5]:
def evaluate_math_dataset(data, model, tokenizer, batch_size = 1, max_new_tokens = 128):
    torch.cuda.empty_cache()

    messages = []

    for entry in tqdm(data, desc="Preparing data"):
        text = []
        text.append({"role": "system", "content": "You are a calculation assistant. You only reply with the result to the equation."})
        text.append({"role": "user", "content": f"{entry['text']}"})
        messages.append(text)

    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, batch_size=batch_size)
    # outputs = pipe(messages, max_new_tokens=max_new_tokens, return_full_text=False)

    outputs = []
    for i in tqdm(range(0, len(messages), batch_size), desc="Generating responses"):
        torch.cuda.empty_cache()
        batch = messages[i:i+batch_size]
        output_batch = pipe(batch, max_new_tokens=max_new_tokens, return_full_text=False)
        outputs.extend(output_batch)

    correct = 0
    total = len(data)
    i = 0

    for entry in tqdm(data, desc="Processing outputs"):
        correct_solution = entry['output']
        if correct_solution in outputs[i][0]['generated_text'].strip():
            correct += 1
        i += 1

    accuracy = correct / total

    return accuracy

In [6]:
def split_math_ds(ds, name_of_method):
    print(f"Splitting Dataset: {name_of_method} in zero, one and two negativs")
    no_neg  = []
    one_neg = []
    two_neg = []
    if name_of_method == "subtraction":
        for entry in tqdm(ds):
            if entry['text'].count("-") > 2:
                two_neg.append(entry)
            elif entry['text'].count("-") < 2:
                no_neg.append(entry)
            else:
                one_neg.append(entry)
    else:
        for entry in tqdm(ds):
            if entry['text'].count("-") > 1:
                two_neg.append(entry)
            elif entry['text'].count("-") < 1:
                no_neg.append(entry)
            else:
                one_neg.append(entry)

    datasets = {"no_neg": Dataset.from_list(no_neg),
                "one_neg": Dataset.from_list(one_neg),
                "two_neg": Dataset.from_list(two_neg)}
    datasets["no_neg"].info.description  = f"{name_of_method} - has no negativs"
    datasets["one_neg"].info.description = f"{name_of_method} - has one negativ"
    datasets["two_neg"].info.description = f"{name_of_method} - has two negativs"
    # for key, value in datasets.items():
    #     value.save_to_disk(f"./datasets/special_cases/{name_of_method}_{key}")
    return datasets

In [7]:
ds = {"addition": math_ds_add, "subtraction": math_ds_sub, "multiplication": math_ds_mul, "division": math_ds_div}
all_sub_sets = []
for key, value in ds.items():
    all_sub_sets.append(split_math_ds(value, key))

Splitting Dataset: addition in zero, one and two negativs


100%|██████████| 50208/50208 [00:01<00:00, 36610.23it/s]


Splitting Dataset: subtraction in zero, one and two negativs


100%|██████████| 49476/49476 [00:01<00:00, 37338.34it/s]


Splitting Dataset: multiplication in zero, one and two negativs


100%|██████████| 50199/50199 [00:01<00:00, 37268.91it/s]


Splitting Dataset: division in zero, one and two negativs


100%|██████████| 50117/50117 [00:01<00:00, 36717.07it/s]


In [9]:
len(max(custom_math_ds['output'], key=len))

9

In [10]:
for ds in all_sub_sets:
    for key, value in ds.items():
        print(f"\033[1m{value.description}\n \033[0m"
              f"{len(value)}")
        print(len(max(value['output'], key=len)))
        print(f"{value.description.split(sep=' ')[0]}_{key}")
    print("")

addition - has no negativs
 12566
4
addition_no_neg
addition - has one negativ
 25101
4
addition_one_neg
addition - has two negativs
 12541
5
addition_two_neg

subtraction - has no negativs
 12224
4
subtraction_no_neg
subtraction - has one negativ
 25021
5
subtraction_one_neg
subtraction - has two negativs
 12231
4
subtraction_two_neg

multiplication - has no negativs
 12520
6
multiplication_no_neg
multiplication - has one negativ
 25125
7
multiplication_one_neg
multiplication - has two negativs
 12554
7
multiplication_two_neg

division - has no negativs
 12431
8
division_no_neg
division - has one negativ
 24993
9
division_one_neg
division - has two negativs
 12693
8
division_two_neg



In [11]:
model_name = 'microsoft/Phi-3-mini-128k-instruct'
baseline_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2", device_map=device)
baseline_tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.90s/it]


In [12]:
accuracy_scores_all = {}
for ds in all_sub_sets:
    accuracy_scores = {}
    for key, value in ds.items():
        print(f"Evaluating:\n"
              f"{value.description}\n"
              f"Number of entries: {len(value)}")
        acc = evaluate_math_dataset(value, baseline_model, baseline_tokenizer, batch_size = 256, max_new_tokens = 16)
        print(f"Accuracy: {acc}\n")
        accuracy_scores[f"{value.description.split(sep=' ')[0]}_{key}"] = acc
    file = open(f'results/special_cases/custom/math_ds_{value.description.split(sep=' ')[0]}_splitted.json', 'w')
    json.dump(accuracy_scores, file)
    file.close()
    print(accuracy_scores)
    accuracy_scores_all[value.description.split(sep=' ')[0]] = accuracy_scores
    file = open('results/special_cases/custom/math_ds_splitted.json', 'w')
    json.dump(accuracy_scores_all, file)
    file.close()

Evaluating:
addition - has no negativs
Number of entries: 12566


Processing outputs: 100%|██████████| 12566/12566 [00:00<00:00, 37417.77it/s]


Accuracy: 0.9574247970714627

Evaluating:
addition - has one negativ
Number of entries: 25101


Processing outputs: 100%|██████████| 25101/25101 [00:00<00:00, 36899.14it/s]


Accuracy: 0.9263774351619457

Evaluating:
addition - has two negativs
Number of entries: 12541


...

Accuracy: 0.15203925669696725

Evaluating:
division - has one negativ
Number of entries: 24993


Processing outputs: 100%|██████████| 24993/24993 [00:00<00:00, 34455.73it/s]


Accuracy: 0.13887888608810467

Evaluating:
division - has two negativs
Number of entries: 12693


Processing outputs: 100%|██████████| 12693/12693 [00:00<00:00, 34551.32it/s]

Accuracy: 0.11533916331836445

{'division_no_neg': 0.15203925669696725, 'division_one_neg': 0.13887888608810467, 'division_two_neg': 0.11533916331836445}


In [14]:
accuracy_scores_all

{'addition': {'addition_no_neg': 0.9574247970714627,
  'addition_one_neg': 0.9263774351619457,
  'addition_two_neg': 0.9177099114903118},
 'subtraction': {'subtraction_no_neg': 0.9396269633507853,
  'subtraction_one_neg': 0.8872547060469206,
  'subtraction_two_neg': 0.8592102035810645},
 'multiplication': {'multiplication_no_neg': 0.1762779552715655,
  'multiplication_one_neg': 0.18606965174129353,
  'multiplication_two_neg': 0.18201370081249005},
 'division': {'division_no_neg': 0.15203925669696725,
  'division_one_neg': 0.13887888608810467,
  'division_two_neg': 0.11533916331836445}}

# list Dataset

In [2]:
# custom_math_list_ds = u.create_math_lists_tasks(amount=200000, 
#                                                 upper_bound=1000,
#                                                 lower_bound=-1000,
#                                                 max_entries=10
#                                                 )
# custom_math_list_ds

Dataset({
    features: ['text', 'output'],
    num_rows: 200000
})

In [3]:
# custom_math_list_ds.save_to_disk("./datasets/custom_math_lists_ds")

Saving the dataset (1/1 shards): 100%|██████████| 200000/200000 [00:00<00:00, 4089809.37 examples/s]


In [2]:
custom_math_list_ds = Dataset.load_from_disk("./datasets/custom_math_lists_ds")
custom_math_list_ds

Dataset({
    features: ['text', 'output'],
    num_rows: 200000
})

In [3]:
# mean = []
# median = []
# min = []
# max = []
# 
# for entry in tqdm(custom_math_list_ds): 
#     if " mean " in entry['text']:
#         mean.append(entry)
#     elif " median " in entry['text']:
#         median.append(entry)
#     elif " max " in entry['text']:
#         max.append(entry)
#     elif " min " in entry['text']:
#         min.append(entry)
#     
# 
# math_list_ds_mean = Dataset.from_list(mean)
# math_list_ds_median = Dataset.from_list(median)
# math_list_ds_min = Dataset.from_list(min)
# math_list_ds_max = Dataset.from_list(max)
# 
# math_list_ds_mean.info.description = (f"Original Dataset: 'Custom Math List Dataset - Lower Bound: -1000 Upper Bound: 1000 Max entries: 10'\n"
#                                      f"Mean Dataset - Subset with only mean operations\n")
# math_list_ds_median.info.description = (f"Original Dataset: 'Custom Math List Dataset - Lower Bound: -1000 Upper Bound: 1000 Max entries: 10'\n"
#                                         f"Median Dataset - Subset with only median operations\n")
# math_list_ds_min.info.description = (f"Original Dataset: 'Custom Math List Dataset - Lower Bound: -1000 Upper Bound: 1000 Max entries: 10\n"
#                                      f"Min Dataset - Subset with only minimum operations\n")
# math_list_ds_max.info.description = (f"Original Dataset: 'Custom Math List Dataset - Lower Bound: -1000 Upper Bound: 1000 Max entries: 10\n"
#                                            f"Max Dataset - Subset with only maximum operations\n")
# 
# math_list_ds_mean.save_to_disk("./datasets/custom/math_list_mean")
# math_list_ds_median.save_to_disk("./datasets/custom/math_list_median")
# math_list_ds_min.save_to_disk("./datasets/custom/math_list_min")
# math_list_ds_max.save_to_disk("./datasets/custom/math_list_max")
# 
# # print(math_ds_addition)
# # print(math_ds_subtraction)
# # print(math_ds_division)
# # print(math_ds_multiplication)

In [4]:
math_list_ds_mean = Dataset.load_from_disk("./datasets/custom/math_list_mean")
math_list_ds_median = Dataset.load_from_disk("./datasets/custom/math_list_median")
math_list_ds_min = Dataset.load_from_disk("./datasets/custom/math_list_min")
math_list_ds_max = Dataset.load_from_disk("./datasets/custom/math_list_max")

In [5]:
def evaluate_math_dataset(data, model, tokenizer, batch_size = 1, max_new_tokens = 128):
    torch.cuda.empty_cache()

    messages = []

    for entry in tqdm(data, desc="Preparing data"):
        text = []
        text.append({"role": "system", "content": "You are a calculation assistant. You only reply with the result number, not with a text."})
        text.append({"role": "user", "content": f"{entry['text']}"})
        messages.append(text)

    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, batch_size=batch_size)
    # outputs = pipe(messages, max_new_tokens=max_new_tokens, return_full_text=False)

    outputs = []
    for i in tqdm(range(0, len(messages), batch_size), desc="Generating responses"):
        torch.cuda.empty_cache()
        batch = messages[i:i+batch_size]
        output_batch = pipe(batch, max_new_tokens=max_new_tokens, return_full_text=False)
        outputs.extend(output_batch)

    correct = 0
    total = len(data)
    i = 0

    for entry in tqdm(data, desc="Processing outputs"):
        correct_solution = entry['output']
        if correct_solution in outputs[i][0]['generated_text'].strip():
            correct += 1
        i += 1

    accuracy = correct / total

    return accuracy, data, outputs

In [6]:
ds = {"mean": math_list_ds_mean, "median": math_list_ds_median, "min": math_list_ds_min, "max": math_list_ds_max}
all_sub_sets = []
for key, value in ds.items():
    all_sub_sets.append(value)

In [7]:
model_name = 'microsoft/Phi-3-mini-128k-instruct'
baseline_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2", device_map=device)
baseline_tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.16s/it]


In [8]:
accuracy_scores_all = {}
a = []
d = []
o = []
for ds in all_sub_sets:
    print(f"Evaluating:\n"
          f"{ds.description}\n"
          f"Number of entries: {len(ds)}")
    acc, da, ou = evaluate_math_dataset(ds, baseline_model, baseline_tokenizer, batch_size = 128, max_new_tokens = 64)
    print(f"Accuracy: {acc}\n")
    accuracy_scores_all[f"{ds.description.split(sep='\n')[1].split(sep=' ')[0]}"] = acc
    # print(acc)
    a.append(acc)
    d.append(da)
    o.append(ou)
    
file = open('results/special_cases/custom/math_list_ds.json', 'w')
json.dump(accuracy_scores_all, file)
file.close()

Evaluating:
Original Dataset: 'Custom Math List Dataset - Lower Bound: -1000 Upper Bound: 1000 Max entries: 10'
Mean Dataset - Subset with only mean operations

Number of entries: 50385


Processing outputs: 100%|██████████| 50385/50385 [00:01<00:00, 36855.65it/s]


Accuracy: 0.026416592239753896

Evaluating:
Original Dataset: 'Custom Math List Dataset - Lower Bound: -1000 Upper Bound: 1000 Max entries: 10'
Median Dataset - Subset with only median operations

Number of entries: 49679


Processing outputs: 100%|██████████| 49679/49679 [00:01<00:00, 36969.01it/s]


Accuracy: 0.01376839308359669

Evaluating:
Original Dataset: 'Custom Math List Dataset - Lower Bound: -1000 Upper Bound: 1000 Max entries: 10
Min Dataset - Subset with only minimum operations

Number of entries: 50015


Processing outputs: 100%|██████████| 50015/50015 [00:01<00:00, 36282.20it/s]


Accuracy: 0.9221433569929022

Evaluating:
Original Dataset: 'Custom Math List Dataset - Lower Bound: -1000 Upper Bound: 1000 Max entries: 10
Max Dataset - Subset with only maximum operations

Number of entries: 49921


Processing outputs: 100%|██████████| 49921/49921 [00:01<00:00, 36662.79it/s]

Accuracy: 0.9008433324652951



In [9]:
accuracy_scores_all

{'Mean': 0.026416592239753896,
 'Median': 0.01376839308359669,
 'Min': 0.9221433569929022,
 'Max': 0.9008433324652951}

In [10]:
a

[0.026416592239753896,
 0.01376839308359669,
 0.9221433569929022,
 0.9008433324652951]

In [11]:
d[0]

Dataset({
    features: ['text', 'output'],
    num_rows: 50385
})

In [12]:
o

[[[{'generated_text': ' 52.5'}],
  [{'generated_text': ' To find the mean value of the list, you sum all the numbers and then divide by the count of the numbers.\n\n\nSum = 209 + (-136) + (-935) + (-939) + (-809) + (-553) +'}],
  [{'generated_text': ' To calculate the mean value of the list, you sum all the numbers and then divide by the count of the numbers.\n\n\nSum of the numbers: 992 + 888 + (-225) + (-839) + 130 + (-400)'}],
  [{'generated_text': ' To find the mean value of the list, you sum all the numbers and then divide by the count of the numbers.\n\n\nMean = (751 + (-524) + 774 + (-794) + (-222)) / 5\n\nMean'}],
  [{'generated_text': ' -687'}],
  [{'generated_text': ' To find the mean value of the list, you sum all the numbers and then divide by the count of the numbers.\n\n\nSum of the numbers: -399 + (-110) + (-677) + (-71) + (-994) + 953'}],
  [{'generated_text': ' To find the mean value of the list, you sum all the numbers and then divide by the count of the numbers.\n\n\